- **Name:** 
- **Term:** 

### Machine Problem 006: Time Response of Second-Order Systems

The [Python Control Systems Library](https://github.com/python-control/python-control) provides tools to simulate and analyze the time response of control systems. In this notebook, we use it to study the **step response** of second-order systems — including how pole locations determine the form of the response, how to compute the **natural frequency** and **damping ratio**, and how to measure key underdamped response specifications.

Consult the [Python Control Systems Documentation](http://python-control.readthedocs.io/en/latest/) for more details.

#### Installation

The [Python Control Systems Library](https://github.com/python-control/python-control) is not a standard part of most Python distributions. Run the following commands once to install the required packages.

To install both the Control Systems library and Slycot in an existing conda environment, run:
`!conda install -c conda-forge control slycot`

In [ ]:
!conda install -c conda-forge slycot
!pip install control
!pip install seaborn

#### Library Usage

The control systems library is designed to work with a simplified syntax where libraries are imported without the standard prefixes.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import control.matlab as control

#### Demonstration

##### General Second-Order Transfer Function

A **second-order system** has a transfer function of the form:

$$G(s) = \frac{b}{s^2 + as + b}$$

Unlike a first-order system where the parameter $a$ only affects the speed of response, **changing the parameters $a$ and $b$ in a second-order system can change the entire form of the response.**

The behavior of the system depends on the **poles** — the roots of the characteristic equation:

$$s^2 + as + b = 0 \quad \Longrightarrow \quad s = \frac{-a \pm \sqrt{a^2 - 4b}}{2}$$

The **discriminant** $\Delta = a^2 - 4b$ determines the nature of the poles and thus the form of the step response:

| Condition | Poles | Response Type |
|---|---|---|
| $a = 0$ | Purely imaginary: $\pm j\omega$ | Undamped (oscillatory) |
| $\Delta > 0$ | Two real, distinct | Overdamped |
| $\Delta = 0$ | Two real, equal | Critically damped |
| $\Delta < 0$ | Complex conjugate | Underdamped (decaying oscillation) |

##### The Four Types of Step Responses

We now demonstrate all four response types by varying the parameter $a$ while keeping $b = 9$.

**Undamped:** $a = 0$ → $G(s) = \dfrac{9}{s^2 + 9}$  
Poles: $s = \pm j3$ (purely imaginary)

**Overdamped:** $a = 9$ → $G(s) = \dfrac{9}{s^2 + 9s + 9}$  
Discriminant: $81 - 36 = 45 > 0$ → two real, distinct poles

**Critically Damped:** $a = 6$ → $G(s) = \dfrac{9}{s^2 + 6s + 9}$  
Discriminant: $36 - 36 = 0$ → two real, equal poles

**Underdamped:** $a = 2$ → $G(s) = \dfrac{9}{s^2 + 2s + 9}$  
Discriminant: $4 - 36 = -32 < 0$ → complex conjugate poles

In [ ]:
b = 9   # fixed numerator / b coefficient

systems = {
    'Undamped  (a=0)':            control.tf([b], [1, 0, b]),
    'Underdamped (a=2)':          control.tf([b], [1, 2, b]),
    'Critically Damped (a=6)':    control.tf([b], [1, 6, b]),
    'Overdamped  (a=9)':          control.tf([b], [1, 9, b]),
}

colors = ['blue', 'green', 'orange', 'red']

plt.figure(figsize=(9, 5))
for (label, G), color in zip(systems.items(), colors):
    poles = control.pole(G)
    print(f'{label}: poles = {np.round(poles, 4)}')
    y, t = control.step(G)
    plt.plot(t, y, color=color, linewidth=2, label=label)

plt.axhline(1, color='gray', linestyle=':', linewidth=1)
plt.xlabel('Time (s)')
plt.ylabel('Output c(t)')
plt.title('Step Responses of Second-Order Systems (b = 9)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

We can also visualize the pole locations for each system on the $s$-plane. Note how the position of the poles relative to the imaginary axis governs the response type.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
markers = ['o', 's', '^', 'D']

for (label, G), color, marker in zip(systems.items(), colors, markers):
    poles = control.pole(G)
    ax.plot(poles.real, poles.imag, marker, markersize=10,
            markerfacecolor='none', markeredgewidth=2,
            color=color, label=label)

ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Real Axis (σ)')
ax.set_ylabel('Imaginary Axis (jω)')
ax.set_title('Pole Locations for Four Response Types')
ax.legend(fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

##### The Canonical (Standard) Form

To quantitatively describe second-order systems, we rewrite the transfer function in **canonical form**:

$$G(s) = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

where:
- $\omega_n$ = **natural frequency** (rad/s) — the oscillation frequency when there is no damping
- $\zeta$ (zeta) = **damping ratio** — a dimensionless measure of how quickly oscillations decay

Comparing with $G(s) = \dfrac{b}{s^2 + as + b}$, we can identify:

$$\omega_n = \sqrt{b} \qquad \zeta = \frac{a}{2\omega_n} = \frac{a}{2\sqrt{b}}$$

The damping ratio $\zeta$ determines the response type:

| $\zeta$ | Response Type |
|---|---|
| $\zeta = 0$ | Undamped |
| $0 < \zeta < 1$ | Underdamped |
| $\zeta = 1$ | Critically Damped |
| $\zeta > 1$ | Overdamped |

The system poles (from the characteristic equation) are:

$$s_{1,2} = -\zeta\omega_n \pm \omega_n\sqrt{\zeta^2 - 1}$$

For the underdamped case ($\zeta < 1$), the poles are complex:

$$s_{1,2} = -\zeta\omega_n \pm j\,\omega_d \qquad \text{where } \omega_d = \omega_n\sqrt{1 - \zeta^2}$$

$\omega_d$ is the **damped natural frequency**.

In [ ]:
# Example: G(s) = 25 / (s^2 + 6s + 25)
G_ex = control.tf([25], [1, 6, 25])
print('Transfer Function G(s):')
print(G_ex)

# Extract parameters from standard form s^2 + as + b
b_coef = 25
a_coef = 6

wn  = np.sqrt(b_coef)          # natural frequency
zeta = a_coef / (2 * wn)       # damping ratio
wd  = wn * np.sqrt(1 - zeta**2) # damped natural frequency

print(f'\nNatural Frequency    ωn  = {wn:.4f} rad/s')
print(f'Damping Ratio        ζ   = {zeta:.4f}')
print(f'Damped Nat. Freq.    ωd  = {wd:.4f} rad/s')

poles = control.pole(G_ex)
print(f'Poles: {np.round(poles, 4)}')

if zeta == 0:
    print('Response type: Undamped')
elif 0 < zeta < 1:
    print('Response type: Underdamped')
elif zeta == 1:
    print('Response type: Critically Damped')
else:
    print('Response type: Overdamped')

##### Underdamped Response Specifications

For an **underdamped** second-order system, the following four specifications quantify transient performance:

| Specification | Symbol | Formula |
|---|---|---|
| Peak Time | $T_p$ | $T_p = \dfrac{\pi}{\omega_d}$ |
| Percent Overshoot | $\%OS$ | $\%OS = e^{-\zeta\pi/\sqrt{1-\zeta^2}} \times 100\%$ |
| Settling Time | $T_s$ | $T_s = \dfrac{4}{\zeta\omega_n}$ |
| Rise Time | $T_r$ | $T_r \approx \dfrac{1.8}{\omega_n}$ |

We compute and annotate these on the step response of $G(s) = \dfrac{25}{s^2 + 6s + 25}$.

In [ ]:
# Compute specifications
Tp  = np.pi / wd                                          # Peak time
OS  = np.exp(-zeta * np.pi / np.sqrt(1 - zeta**2)) * 100 # % Overshoot
Ts  = 4 / (zeta * wn)                                    # Settling time
Tr  = 1.8 / wn                                           # Rise time (approx)

print(f'Peak Time         Tp  = {Tp:.4f} s')
print(f'Percent Overshoot %OS = {OS:.2f}%')
print(f'Settling Time     Ts  = {Ts:.4f} s')
print(f'Rise Time (approx) Tr = {Tr:.4f} s')

# Step response
y, t = control.step(G_ex)
c_final = y[-1]
c_peak  = c_final * (1 + OS / 100)

plt.figure(figsize=(9, 5))
plt.plot(t, y, 'b-', linewidth=2, label='c(t)')

# Final value
plt.axhline(c_final, color='gray', linestyle=':', linewidth=1,
            label=f'Final value = {c_final:.2f}')

# Peak time
plt.axvline(Tp, color='red', linestyle='--',
            label=f'Tp = {Tp:.3f} s')
plt.axhline(c_peak, color='red', linestyle=':', alpha=0.4)

# Rise time
plt.axvline(Tr, color='orange', linestyle='--',
            label=f'Tr ≈ {Tr:.3f} s')

# Settling time with 2% band
plt.axvline(Ts, color='green', linestyle='--',
            label=f'Ts = {Ts:.3f} s')
plt.axhline(0.98 * c_final, color='green', linestyle=':', alpha=0.4)
plt.axhline(1.02 * c_final, color='green', linestyle=':', alpha=0.4)

# Overshoot annotation
plt.annotate(f'%OS = {OS:.2f}%',
             xy=(Tp, c_peak), xytext=(Tp + 0.3, c_peak + 0.05),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=9, color='red')

plt.xlabel('Time (s)')
plt.ylabel('Output c(t)')
plt.title('Underdamped Step Response with Specifications\n'
          f'G(s) = 25/(s²+6s+25),  ωn={wn:.1f} rad/s,  ζ={zeta:.2f}')
plt.legend(loc='upper right', fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

##### Effect of Damping Ratio on Underdamped Response

For a fixed natural frequency $\omega_n = 5$ rad/s, we compare the step response for several values of $\zeta$ in the underdamped range ($0 < \zeta < 1$). A smaller $\zeta$ produces more oscillation and greater overshoot.

In [ ]:
wn_fixed = 5
zeta_values = [0.1, 0.3, 0.5, 0.7, 0.9]
colors_zeta = ['navy', 'blue', 'green', 'orange', 'red']

plt.figure(figsize=(9, 5))

for z, color in zip(zeta_values, colors_zeta):
    G = control.tf([wn_fixed**2],
                   [1, 2*z*wn_fixed, wn_fixed**2])
    y, t = control.step(G)
    plt.plot(t, y, color=color, linewidth=2, label=f'ζ = {z}')

plt.axhline(1, color='gray', linestyle=':', linewidth=1)
plt.xlabel('Time (s)')
plt.ylabel('Output c(t)')
plt.title(f'Effect of Damping Ratio on Step Response  (ωn = {wn_fixed} rad/s)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'\n{"ζ":>6}  {"ωd (rad/s)":>12}  {"Tp (s)":>10}  {"%OS":>8}  {"Ts (s)":>10}')
print('-' * 55)
for z in zeta_values:
    wd_z  = wn_fixed * np.sqrt(1 - z**2)
    Tp_z  = np.pi / wd_z
    OS_z  = np.exp(-z * np.pi / np.sqrt(1 - z**2)) * 100
    Ts_z  = 4 / (z * wn_fixed)
    print(f'{z:>6.1f}  {wd_z:>12.4f}  {Tp_z:>10.4f}  {OS_z:>8.2f}  {Ts_z:>10.4f}')

---
#### Exercises

Apply what you have learned from the demonstration above to solve the following problems. Add your code in the blank cells provided. Show all computed values and plots.

##### Exercise 1

For each of the following transfer functions, determine — **by inspection of the poles** — the form of the step response (undamped, underdamped, critically damped, or overdamped). Then verify by plotting all three step responses on a **single figure**.

$$G_A(s) = \frac{16}{s^2 + 16} \qquad G_B(s) = \frac{16}{s^2 + 8s + 16} \qquad G_C(s) = \frac{16}{s^2 + 3s + 16}$$

*(a)* For each system, print the **poles** and state the **response type**.  
*(b)* Plot all three step responses on the same figure with a legend.  
*(c)* Do the plots confirm your initial inspection? Explain briefly.

In [ ]:
# Exercise 1 — your code here


*Write your explanation for part (c) here:*



##### Exercise 2

Given the transfer function:

$$G(s) = \frac{36}{s^2 + 4.2s + 36}$$

*(a)* Identify the **natural frequency** $\omega_n$ and **damping ratio** $\zeta$.  
*(b)* Compute the **damped natural frequency** $\omega_d$.  
*(c)* State the **response type** based on $\zeta$.  
*(d)* Compute the **peak time** $T_p$, **percent overshoot** $\%OS$, **settling time** $T_s$, and **rise time** $T_r$.  
*(e)* Plot the step response and **annotate** $T_p$, $T_s$, and the 2% settling band.

In [ ]:
# Exercise 2 — your code here


##### Exercise 3

A second-order system is required to meet the following specifications:

- **Natural frequency:** $\omega_n = 8$ rad/s  
- **Damping ratio:** $\zeta = 0.4$

*(a)* Write the **transfer function** $G(s)$ in canonical form.  
*(b)* Compute $T_p$, $\%OS$, $T_s$, and $T_r$.  
*(c)* Plot the **step response** and annotate $T_p$, $T_s$, and the percent overshoot.  
*(d)* If the damping ratio is increased to $\zeta = 0.7$ (keeping $\omega_n = 8$), how do the specifications change? Plot both responses on the same figure and compare.

In [ ]:
# Exercise 3 — your code here


*Write your comparison for part (d) here:*



##### Exercise 4

For each of the following transfer functions, find the **natural frequency**, **damping ratio**, and **response type**. Then use these values to compute $T_p$, $\%OS$, $T_s$, and $T_r$ **only for the underdamped cases**. Present your results in a printed summary table.

$$G_1(s) = \frac{12}{s^2 + 8s + 12} \qquad G_2(s) = \frac{16}{s^2 + 2s + 16} \qquad G_3(s) = \frac{4}{s^2 + 4s + 4}$$

In [ ]:
# Exercise 4 — your code here
